# Node 04 · 对接（1,765 配体 × 3 靶点 × 5 种子 = 26,475 次）· 控制台

本 notebook 是 Node 04 的【唯一操作入口】。它不含任何科学逻辑 —— 只调度与核验。

## 为什么科学代码【不】搬进 notebook

| 保持为 .py 的文件 | 理由 |
|---|---|
| `core/node04_docking.py` | 库。Slurm 计算节点上的 worker 要 `import` 它 —— 搬进 notebook 会切断执行路径 |
| `core/node04a_receptor_prep.py` | 同上（受体侧，已冻结） |
| `core/node04_worker.py` | Slurm array 任务入口，由 sbatch 直接调用 |
| `core/node04_array.sbatch` | Slurm 批处理脚本 |
| `core/node04a_verify.py`、`core/node04_verify.py` | 独立核验器。其价值【正在于】与库不共享代码；本 notebook 以子进程调用，绝不 import |

**冻结参数只存在于 `node04_docking.py` 一处**：`exhaustiveness=32`、`num_modes=9`、`--cpu 1`、
种子 `(42,43,44,45,46)`、`BBS_RULE`、`HEAVY_ATOM_MIN=12`。改动必须升版本号。
notebook 里【不得】出现任何科学取值的副本 —— 两份副本必然漂移。

## 执行顺序（闸门，非流程）

    A 上游闸门 → B 工作清单 → C 自对接阳性对照 ┐
                                              ├→ 均通过 → E 全量 → F 汇总 → G 独立核验
                              D 试点 + 分析   ┘

C 与 D 是**闸门**：任一靶点未在 2 Å 内复现晶体姿势，或 Δ_stereo 落在 σ_seed 之内，
都必须先处理再谈全量。提交作业的单元格均设 `SUBMIT = False` 保险，需显式打开。


## Cell 1 · 配置（重启安全）

In [ ]:
# ── Cell 1 · 配置 ────────────────────────────────────────────
import os, sys, json, subprocess
from pathlib import Path

PROJECT_ROOT = Path("/mnt/hpc/home/wanghongmei/lang2").resolve()
PY    = Path("/mnt/hpc/home/wanghongmei/anaconda3/envs/md_env/bin/python")
VINA  = Path("/mnt/hpc/home/wanghongmei/apps/vina/vina_1.2.7_linux_x86_64")
SBATCH = PROJECT_ROOT / "core" / "node04_array.sbatch"
OUT    = PROJECT_ROOT / "results" / "04_docking"

# JupyterHub 不把 md_env/bin 放进 PATH；Slurm 作业也不继承 conda 环境。
_ENV_BIN = str(Path(sys.executable).parent)
if _ENV_BIN not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = _ENV_BIN + os.pathsep + os.environ["PATH"]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

N_TASKS_FULL, N_TASKS_PILOT = 353, 32      # 1765 = 5 × 353；MaxArraySize=1001
SUBMIT = False                              # ★ 提交作业的保险；需要时改 True

print("root  :", PROJECT_ROOT)
print("python:", sys.executable)
print("vina  :", VINA, "存在:", VINA.is_file())
for f in ("node04_docking.py","node04_worker.py","node04_array.sbatch",
          "node04_verify.py","node04a_verify.py","node04_redock_control.py",
          "node04_pilot_analysis.py","node04_collect.py"):
    p = PROJECT_ROOT/"core"/f
    print(f"  {'✅' if p.is_file() else '❌'} core/{f}")

## Cell 2 · 导入库 + 冻结参数一览

In [ ]:
# ── Cell 2 · 导入 + 参数 ─────────────────────────────────────
import importlib
from core import node04_docking as N4
importlib.reload(N4)

print("node04_docking:", N4.NODE04_VERSION)
print(f"  exhaustiveness={N4.EXHAUSTIVENESS}  num_modes={N4.NUM_MODES}  cpu={N4.VINA_CPU}")
print(f"  seeds={N4.SEEDS}")
print(f"  BBS_RULE={N4.BBS_RULE}")
print(f"  {N4.BBS_NOTE}")
print(f"  HEAVY_ATOM_MIN={N4.HEAVY_ATOM_MIN}")

## Cell 3 · 闸门 A —— 上游哈希核验（fail-closed）

重算 1,765 个配体 PDBQT + 3 个受体 PDBQT 的 sha256，对照 Node 03 与 Node 04a 的 manifest。
任一不符即 raise。约 10–30 秒。**任何异常都是真实发现，不要绕过。**

In [ ]:
# ── Cell 3 · 上游闸门 + pool_class ───────────────────────────
import pandas as pd

gate, recs, m4a = N4.verify_upstream(
    PROJECT_ROOT/"results/03_ligand_pdbqt/Node03_manifest.json",
    PROJECT_ROOT/"results/03_ligand_pdbqt/identity_manifests_node03.jsonl",
    OUT/"receptors/Node04a_receptor_manifest.json")
pool_class, charge, pc_stats = N4.derive_pool_class(
    PROJECT_ROOT/"results/02_master_molecules/identity_manifests.jsonl",
    node00_csv=PROJECT_ROOT/"results/00_scope/all_candidates_v3_start.csv",
    pandas_mod=pd)

print("闸门:", gate["assertion"])
print(f"  配体 {gate['n_ligands_verified']} | 受体 {gate['n_receptors_verified']}"
      f" | node03 v{gate['node03_version']} | node04a v{gate['node04a_version']}")
print(f"  pool_class: candidate {pc_stats['n_candidate']} / fragment {pc_stats['n_fragment']}")
print(f"  逐分子交叉核对: {pc_stats['cross_check_vs_node00']}")
for pid, m in sorted(m4a["receptors"].items()):
    print(f"  {pid}: 盒心 {m['box_center']} | 二联体 {m['catalytic_dyad']}")

## Cell 4 · 工作清单（全量 26,475 + 试点 2,100）

试点前 5 个种子与全量【完全相同】；`run_one` 幂等，故全量阶段会跳过 —— 试点不浪费机时。

In [ ]:
# ── Cell 4 · 构建两份工作清单（对运行中的作业安全）──────────
# ★ 危险点：353 个 array 任务【正在读】node04_work_manifest.jsonl。
#   非原子覆盖会让尚未启动的任务读到截断文件或错位的切片。
#   故：内容相同 → 跳过写盘；内容不同且有作业在跑 → 拒绝并报错；
#       确需写盘 → 先写 .tmp 再 os.replace（POSIX 原子替换）。
import random, hashlib, os
PILOT_SEEDS   = N4.SEEDS + (47, 48, 49, 50, 51)
PILOT_PARENTS = 25

def _running_n04():
    r = subprocess.run(["squeue","-u",os.environ.get("USER","wanghongmei"),
                        "-h","-o","%i %j %T"], capture_output=True, text=True)
    return [l for l in r.stdout.splitlines() if "prism_n04" in l]

def safe_write(path, text):
    path = Path(path)
    new = hashlib.sha256(text.encode()).hexdigest()
    if path.is_file():
        old = hashlib.sha256(path.read_bytes()).hexdigest()
        if old == new:
            print(f"  = {path.name} 内容未变，跳过写盘 (sha {new[:12]}…)"); return new
        jobs = _running_n04()
        if jobs:
            raise RuntimeError(
                f"拒绝覆盖 {path.name}：内容已变，但有 Node 04 作业在运行：\n  "
                + "\n  ".join(jobs) + "\n作业正在读这份清单，覆盖会导致切片错位。"
                "\n请先 scancel 或等其结束。")
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8"); os.replace(tmp, path)   # 原子替换
    print(f"  ✓ {path.name} 已写 (sha {new[:12]}…)")
    return new

rows = N4.build_work_manifest(recs, m4a, OUT, seeds=N4.SEEDS)
wm = OUT/"node04_work_manifest.jsonl"; wm.parent.mkdir(parents=True, exist_ok=True)
safe_write(wm, "\n".join(json.dumps(r, ensure_ascii=False) for r in rows)+"\n")

byp = {}
for r in recs: byp.setdefault(r.get("parent_id") or r["name"], []).append(r)
multi = {p: v for p, v in byp.items() if len(v) > 1}
random.seed(20260728)
pick = sorted(random.sample(sorted(multi), min(PILOT_PARENTS, len(multi))))
pilot_recs = [x for p in pick for x in multi[p]]
prows = N4.build_work_manifest(pilot_recs, m4a, OUT, seeds=PILOT_SEEDS)
pwm = OUT/"node04_pilot_manifest.jsonl"
safe_write(pwm, "\n".join(json.dumps(r, ensure_ascii=False) for r in prows)+"\n")

print(f"\n全量 {len(rows)} 行  分块 {N_TASKS_FULL}×{len(rows)//N_TASKS_FULL}")
print(f"试点 {len(prows)} 行  ({len(pick)} 母体 / {len(pilot_recs)} 配体)")

## Cell 5 · 闸门 C —— 自对接阳性对照

X77→6W63、0EN→3V3M、R30→4YLU，用同一冻结盒与受体。这是唯一对照**流水线之外**事物的检查。

**前置**：三个键级模板需手动放好（gpu01 无外网）：
`results/04_docking/redock_control/{6W63/X77,3V3M/0EN,4YLU/R30}_ideal.sdf`

判据：最优得分姿势 RMSD ≤ 2 Å（对称性感知、**不叠合**）。两列 RMSD 都要看 ——
`RMSD(top)` 检验打分函数是否把晶体姿势排第一；`RMSD(min)` 只说明搜索找到过它。

In [ ]:
# ── Cell 5 · 自对接阳性对照 ──────────────────────────────────
# ★ 本单元格要跑 15 次 Vina（3 配体 × 5 种子，约 45 分钟单核）。
#   gpu01 是共享头节点 —— 必须经 srun 丢到计算节点，不能在头节点上跑。
cmd = ["srun", "-p", "cpuPartition", "-n1", "-c4",
       str(PY), "core/node04_redock_control.py"]
print(" ".join(cmd), "\n")
r = subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_ROOT)
print(r.stdout[-4000:])
if r.returncode != 0:
    print("STDERR:\n", r.stderr[-2000:])
    print("\n❌ 阳性对照未通过 —— 在查明原因前不要提交全量。")

## Cell 6 · 闸门 D —— 提交试点（约 12 分钟 / 512 核）

In [ ]:
# ── Cell 6 · 提交试点 ────────────────────────────────────────
(PROJECT_ROOT/"logs").mkdir(exist_ok=True)          # sbatch 相对提交目录解析 --output
cmd = ["sbatch", f"--array=0-{N_TASKS_PILOT-1}%{N_TASKS_PILOT}",
       f"--export=ALL,WM={pwm},NT={N_TASKS_PILOT}", str(SBATCH)]
print(" ".join(cmd))
if SUBMIT:
    print(subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_ROOT).stdout)
else:
    print("[SUBMIT=False] 未提交。确认无误后把 Cell 1 的 SUBMIT 改 True 再跑本单元格。")

## Cell 7 · 进度监控（可反复运行）

In [ ]:
# ── Cell 7 · 监控（可反复运行）──────────────────────────────
import time
q = subprocess.run(["squeue","-u",os.environ.get("USER","wanghongmei"),
                    "-o","%.10i %.9P %.8T %.11M %.5C %.20R"],
                   capture_output=True, text=True).stdout
print(q)
n_runs = len(list((OUT/"runs").rglob("*.json"))) if (OUT/"runs").is_dir() else 0
tdir = OUT/"tasks"/f"node04_work_manifest__n{N_TASKS_FULL}"
n_tasks = len(list(tdir.glob("task_*.json"))) if tdir.is_dir() else 0
TARGET = 26475
print(f"run JSON : {n_runs:>6} / {TARGET}  ({n_runs/TARGET*100:5.1f}%)")
print(f"task 记录: {n_tasks:>6} / {N_TASKS_FULL}")
errs = [p for p in (PROJECT_ROOT/"logs").glob("n04_*.err") if p.stat().st_size > 0]
print(f"非空 stderr: {len(errs)}" + (f"  {[p.name for p in errs[:5]]}" if errs else "  ✅"))
# 粗略剩余时间：以本次会话两次调用之间的增量估算
_prev = globals().get("_mon_prev")
if _prev and n_runs > _prev[1]:
    rate = (n_runs - _prev[1]) / (time.time() - _prev[0])      # runs/s
    print(f"速率 {rate*3600:.0f} 次/小时 → 预计剩余 {(TARGET-n_runs)/rate/3600:.1f} h")
else:
    print("（再运行一次本单元格可估算速率与剩余时间）")
globals()["_mon_prev"] = (time.time(), n_runs)

## Cell 8 · 试点分析 —— 三个答案

1. **成本**：实测均耗时 → 全量墙钟外推（取代单分子基准）
2. **代表性**：试点 TORSDOF 分布 vs 全池 —— 决定外推可否采信
3. **科学**：σ_seed vs Δ_stereo —— labile-N 差向异构体扩展是否值得做

不完整则拒绝分析（重提交同一 array 即补跑，幂等）。

In [ ]:
# ── Cell 8 · 试点分析 ────────────────────────────────────────
r = subprocess.run([str(PY), "core/node04_pilot_analysis.py"],
                   capture_output=True, text=True, cwd=PROJECT_ROOT)
print(r.stdout)
if r.returncode != 0: print("STDERR:\n", r.stderr[-1500:])

## Cell 9 · 全量（约 2.6 小时 / 512 核）

**前置**：Cell 5 与 Cell 8 均已通过。幂等可续跑 —— 中断后原样重提交即可。

In [ ]:
# ── Cell 9 · 提交全量 ────────────────────────────────────────
jobs = _running_n04()
if jobs:
    print("⚠ 已有 Node 04 作业在运行，拒绝重复提交：")
    for j in jobs[:5]: print("   ", j)
    print(f"   （共 {len(jobs)} 个）")
else:
    cmd = ["sbatch", f"--array=0-{N_TASKS_FULL-1}%32",
           f"--export=ALL,WM={wm},NT={N_TASKS_FULL}", str(SBATCH)]
    print(" ".join(cmd))
    if SUBMIT:
        print(subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_ROOT).stdout)
    else:
        print("[SUBMIT=False] 未提交。")

## Cell 10 · 汇总 —— BBS 表 + manifest

顺序刻意：断言完整 → 算 BBS → 按 `pool_class × charge` 分层 → **然后**施加
【预先登记】的门槛规则（在看到哪些分子被选中之前就已写死，防循环论证）。

In [ ]:
# ── Cell 10 · 汇总 ───────────────────────────────────────────
r = subprocess.run([str(PY), "core/node04_collect.py"],
                   capture_output=True, text=True, cwd=PROJECT_ROOT)
print(r.stdout)
if r.returncode != 0: print("STDERR:\n", r.stderr[-1500:])

## Cell 11 · 独立核验（与库不共享代码）

重算 Vina 与清单哈希、重算每个姿势文件哈希、**从姿势 PDBQT 重新解析亲和力**
（不采信 run JSON）、重算各靶点均值、以最不利靶点重新推导 BBS 并与 CSV 比对。

In [ ]:
# ── Cell 11 · 独立核验 ───────────────────────────────────────
for script, arg in (("core/node04a_verify.py", str(OUT/"receptors/Node04a_receptor_manifest.json")),
                    ("core/node04_verify.py",  str(OUT/"Node04_manifest.json"))):
    if not Path(arg).is_file():
        print(f"[跳过] {script}: {arg} 尚不存在"); continue
    r = subprocess.run([str(PY), script, arg], capture_output=True, text=True, cwd=PROJECT_ROOT)
    print(r.stdout[-2500:]); print(f"exit={r.returncode}\n" + "-"*70)

## Cell 12 · manifest 预览 + 待定事项

In [ ]:
# ── Cell 12 · manifest 预览 ──────────────────────────────────
mp = OUT/"Node04_manifest.json"
if mp.is_file():
    m = json.loads(mp.read_text(encoding="utf-8"))
    print(json.dumps({k: m[k] for k in ("module","version","created_at_utc","docking",
                                        "bbs","threshold_rule","threshold_decision")},
                     indent=2, ensure_ascii=False)[:3000])
else:
    print("尚未生成 —— 先跑 Cell 10。")

---
## 仍待人决定（不是代码问题）

1. **催化二联体质子化** — Meeko 按标准态模板给出中性 His41(HIE) + 中性 Cys 硫醇，三受体一致。
   对非共价对接是可辩护的常规选择，但部分 Mpro 研究采用带电硫醇盐/咪唑鎓离子对。
   须张老师定夺，且写进 Methods。若改，`-n A:41=HIE` / 模板覆盖，三受体同等施加。
2. **MD 门槛的电荷处理** — 规则已预先登记于 `node04_collect.py`。真正的危险是**假阴性**：
   被虚高的带电分子挤到门槛下的优质中性候选永不进 MD，故该偏差永不被观察到。
   Cell 10 的 `threshold_decision` 会给出实测偏移与 σ_seed，据此执行既定规则。
3. **labile-N 差向异构体扩展**（162 分子，+11% 机时）— 由 Cell 8 的 Δ_stereo/σ_seed 比值决定。
   注意：试点用的是立体异构体，是差向异构体的**替代指标**，其作用是给出对接噪声的界。
4. **6W63 盒中心** — 已取共晶配体 X77 质心（距二联体中点 4.75 Å）；如需改回固定参考值再议。

## 整理事项
`core/node04a_receptor_prepV1.py` 是被取代的旧版，请移出 `core/`。
